# NIH metrics

This notebook uses information extracted from [NIH Exporter](https://reporter.nih.gov/exporter) to identify NIH-funded users of PhysioNet.

## Setup

In [128]:
import os
from pathlib import Path

import pandas as pd

from twentyfiveyears.nih import (combine_exporter_tables, get_physionet_users, get_investigators, get_authors, link_users)

In [130]:
# Set the base path
base_path = os.path.join("..", "data")

## Load map of Person IDs

All users are assigned a unique `person_id`.

In [131]:
# Load the map of Person IDs
path = os.path.join(base_path, 'handcrafted', 'person_id_lookup.csv')
person_map = pd.read_csv(path)
person_map.head(3)

,person_id,physionet_id
0,100000000,2
1,100000001,6
2,100000002,8


## Load PhysioNet dataset

Load a dataset containing the list of PhysioNet users

In [155]:
# Load DataFrame of PhysioNet users
path = os.path.join(base_path, 'physionet', 'users.csv')
physionet_users = get_physionet_users(path, person_map)
physionet_users.head(3)

,person_id,physionet_name
0,100000000,felipe t fábregas
1,100000001,tom pollard
2,100000002,benjamin moody


## Load Principal Investigators of NIH projects

Load a list of Principal Investigators

In [133]:
# Load the NIH project data
path = os.path.join(base_path, 'nih', 'exporter', 'projects')
projects = combine_exporter_tables(path, "RePORTER_PRJ_C_FY", start_year=1995)
projects.head(3)

,APPLICATION_ID,ACTIVITY,ADMINISTERING_IC,APPLICATION_TYPE,ARRA_FUNDED,AWARD_NOTICE_DATE,BUDGET_START,BUDGET_END,CFDA_CODE,CORE_PROJECT_NUM,...,SUBPROJECT_ID,SUFFIX,SUPPORT_YEAR,TOTAL_COST,TOTAL_COST_SUB_PROJECT,OPPORTUNITY NUMBER,FUNDING_MECHANISM,ORG_IPF_CODE,DIRECT_COST_AMT,INDIRECT_COST_AMT
0,2056372,A03,AH,1,NaN,1995-05-19T00:00:00,07/01/1995,06/30/1996,NaN,A03AH001117,...,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2056373,A03,AH,1,NaN,1995-05-19T00:00:00,07/01/1995,06/30/1996,NaN,A03AH001118,...,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2056374,A03,AH,1,NaN,1995-05-19T00:00:00,07/01/1995,06/30/1996,NaN,A03AH001119,...,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [134]:
# Get the names of Principal Investigators
investigators = get_investigators(projects)
investigators[0:3]

['eli i capilouto', 'abdelmonem a afifi', 'richard h hart']

## Load authors of publications linked to NIH projects

Load a list of authors linked to NIH projects

In [135]:
# Load the NIH publications data
path = os.path.join(base_path, 'nih', 'exporter', 'publications')
publications = combine_exporter_tables(path, "RePORTER_PUB_C_", start_year=1995)
publications.head(3)

,AFFILIATION,AUTHOR_LIST,COUNTRY,ISSN,JOURNAL_ISSUE,JOURNAL_TITLE,JOURNAL_TITLE_ABBR,JOURNAL_VOLUME,LANG,PAGE_NUMBER,PMC_ID,PMID,PUB_DATE,PUB_TITLE,PUB_YEAR
0,"Department of Fisheries and Wildlife, Oregon S...","Curtis, L R; Zhang, Q; el-Zahr, C; Carpenter, ...",UNITED STATES,0272-0590,1,Fundamental and applied toxicology : official...,Fundam Appl Toxicol,25,eng,146-53,NaN,7601322,1995 Apr,Temperature-modulated incidence of aflatoxin B...,1995
1,"Department of Biology, Boston University, Mass...","Loechler, E L",UNITED STATES,0899-1987,4,Molecular carcinogenesis.,Mol Carcinog,13,eng,213-9,NaN,7646760,1995 Aug,How are potent bulky carcinogens able to induc...,1995
2,Department of Pharmacology and Toxicology Scho...,"Carlson, G P; Olson, R M",AUSTRALIA,1039-9712,1,Biochemistry and molecular biology internation...,Biochem Mol Biol Int,37,eng,65-71,NaN,8653089,1995 Sep,Comparison of the metabolism of alcohols by ra...,1995


In [136]:
# Get the names of authors
authors = get_authors(publications)
authors[0:3]

['l r curtis', 'q zhang', 'c el-zahr']

## Match NIH listed people to PhysioNet users

Attempt to match people between the two sources

In [156]:
# Match NIH Principal Investigators to PhysioNet users
# Set limit for testing
limit = None
physionet_users = link_users(physionet_users, investigators, match_group="investigators", limit=limit)

100%|██████████| 9/9 [00:00<00:00, 34.10it/s]


Finished in 1.8564810752868652 seconds


In [157]:
# Match NIH authors to PhysioNet users
physionet_users = link_users(physionet_users, authors, match_group="authors", limit=limit)

100%|██████████| 9/9 [00:01<00:00,  5.41it/s]


Finished in 9.222126007080078 seconds


In [158]:
physionet_users.head(5)

,person_id,physionet_name,matched_investigator_score,matched_investigator_name,matched_author_score,matched_author_name
0,100000000,felipe t fábregas,0.898643,felipe fregni,0.911230,felipe berg
1,100000001,tom pollard,0.898232,tom lloyd,1.000000,tom pollard
2,100000002,benjamin moody,0.956044,benjamin moon,1.000000,benjamin moody
3,100000003,alistair johnson,0.930357,alistair john cochran,1.000000,alistair johnson
4,100000004,julian e ishii-rousseau,0.878882,julia siri suh,0.904348,julia e sosa


Save the results

## Save the results

In [159]:
# Save the results
save_path = os.path.join(base_path, 'physionet_users_nih_funded.csv')
path = Path(save_path)

# Convert to a path that works on the current OS
normalized_path = path.as_posix() if path.drive else Path(*path.parts).resolve()

# Output the merged DataFrame or save it to a file
physionet_users.to_csv(normalized_path, index=False)